## Few-Shot Transfer under Modular Distribution Shift

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import leap
import numpy as np
import scipy
from LiLY.datasets.sim_dataset import TimeVaryingDataset
from LiLY.modules.modular import ModularShifts
from LiLY.modules.metrics.correlation import correlation
import random
import seaborn as sns
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from leap.tools.utils import load_yaml
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
data = TimeVaryingDataset(directory = '/srv/data/lily/data', 
                          transition='pnl_modular_5',
                          dataset='source')
num_validation_samples = 2500
train_data, val_data = random_split(data, [len(data)-num_validation_samples, num_validation_samples])
train_loader = DataLoader(train_data, batch_size=2560, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_data, batch_size=16, shuffle=False, pin_memory=True)

In [ ]:
cfg = load_yaml('../LiLY/configs/modular_5.yaml')

In [ ]:
model = ModularShifts.load_from_checkpoint(checkpoint_path='/srv/data/lily/log/weiran/modular_5/lightning_logs/version_25/checkpoints/epoch=54-step=168815.ckpt',
                      input_dim=cfg['VAE']['INPUT_DIM'],
                      length=cfg['VAE']['LENGTH'],
                      obs_dim=cfg['SPLINE']['OBS_DIM'],
                      dyn_dim=cfg['VAE']['DYN_DIM'],
                      lag=cfg['VAE']['LAG'],
                      nclass=cfg['VAE']['NCLASS'],
                      hidden_dim=cfg['VAE']['ENC']['HIDDEN_DIM'],
                      dyn_embedding_dim=cfg['VAE']['DYN_EMBED_DIM'],
                      obs_embedding_dim=cfg['SPLINE']['OBS_EMBED_DIM'],
                      trans_prior=cfg['VAE']['TRANS_PRIOR'],
                      lr=cfg['VAE']['LR'],
                      infer_mode=cfg['VAE']['INFER_MODE'],
                      bound=cfg['SPLINE']['BOUND'],
                      count_bins=cfg['SPLINE']['BINS'],
                      order=cfg['SPLINE']['ORDER'],
                      beta=cfg['VAE']['BETA'],
                      gamma=cfg['VAE']['GAMMA'],
                      sigma=cfg['VAE']['SIMGA'],
                      decoder_dist=cfg['VAE']['DEC']['DIST'],
                      correlation=cfg['MCC']['CORR'])

In [ ]:

batch = next(iter(train_loader))
batch_size = batch['xt'].shape[0]

In [ ]:
latent_size = 9

In [ ]:
z, mu, logvar = model.forward(batch)
mu = mu.view(batch_size, -1, latent_size)
A = mu[:,2,:].detach().cpu().numpy()
B = batch['yt'][:,2,:].detach().cpu().numpy()
C = np.zeros((latent_size,latent_size))
for i in range(latent_size):
    C[i] = -np.abs(np.corrcoef(B, A, rowvar=False)[i,latent_size:])
from scipy.optimize import linear_sum_assignment
row_ind, col_ind = linear_sum_assignment(C)
A = A[:, col_ind]
mask = np.ones(latent_size)
for i in range(latent_size):
    if np.corrcoef(B, A, rowvar=False)[i,latent_size:][i] > 0:
        mask[i] = -1
print("Permutation:",col_ind)
print("Sign Flip:", mask)

In [ ]:
figure_path = '${PROJECT_ROOT}/figs/'

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
with PdfPages(figure_path + '/mcc_np.pdf') as pdf:
    fig = plt.figure(figsize=(4,4))
    sns.heatmap(-C, vmin=0, vmax=1, annot=True, fmt=".2f", linewidths=.5, cbar=False, cmap='Greens')
    plt.xlabel("Estimated latents ") 
    plt.ylabel("True latents ") 
    plt.title("MCC=%.3f"%np.abs(C[row_ind, col_ind]).mean());
    pdf.savefig(fig, bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(3,3, figsize=(3,3))
for i in range(9):
    row = i // 3
    col = i % 3
    ax = axs[row,col]
    ax.scatter(B[:,i], A[:,i], s=4, color='b', alpha=0.25)
    ax.axis('off')
#     ax.set_xlabel('Ground truth latent')
#     ax.set_ylabel('Estimated latent')
#     ax.grid('..')
# fig.tight_layout()
fig.supxlabel('True latents')
fig.supylabel('Estimated latents')

In [ ]:
NL + Generalized Gaussian Noise

In [ ]:
theta_dyn = np.zeros((5,2))
for d_idx in range(5):
    W2 = np.load('/srv/data/lily/data/pnl_modular_5/source/W2_%d.npy'%d_idx)
    theta_dyn[d_idx, 0] = W2[1,2]
    theta_dyn[d_idx, 1] = W2[3,4]

In [ ]:
theta_hat = model.dyn_embed_func(torch.LongTensor([0,1,2,3,4])).detach().numpy()

In [ ]:
from matplotlib.patches import FancyArrowPatch 

In [ ]:
from matplotlib.patches import FancyArrowPatch 

In [ ]:
theta_hat = theta_hat.detach().numpy()
theta_dyn = theta_dyn.detach().numpy()

fig, ax = plt.subplots(figsize=(3,3))
ax.set_aspect("equal")
ax.plot(theta_hat[:,0], theta_hat[:,1], color='red', label='Estimated')
ax.plot(theta_dyn[:,0], theta_dyn[:,1], color='gray', label='Matrix entries')
arrow(theta_hat[:,0], theta_hat[:,1], ax, 3)
arrow(theta_dyn[:,0], theta_dyn[:,1], ax, 3)
plt.tight_layout()
plt.legend()
plt.grid()
plt.xlabel(r'$\theta_{k1}^{dyn}$')
plt.ylabel(r'$\theta_{k2}^{dyn}$')
plt.title('Trace of change factors')
def arrow(x,y,ax,n):
    d = len(x)//(n+1)    
    ind = np.arange(d,len(x),d)
    for i in ind:
        ar = FancyArrowPatch ((x[i-1],y[i-1]),(x[i],y[i]), 
                              arrowstyle='->', mutation_scale=20)
        ax.add_patch(ar)

In [ ]:
np.linalg.inv(theta_hat.T @ theta_hat) @ (theta_hat.T @ theta_dyn)

In [ ]:
R = np.random.rand(2,2)

In [ ]:
(theta_hat - theta_dyn @ R)

In [ ]:
5,2 

In [ ]:
theta_dyn.shape

In [ ]:
rotation

In [ ]:
rotation.grad

In [ ]:
rotation.data.add_(-1, rotation.grad)

In [ ]:
rotation.grad

In [ ]:
rotation.grad[0]

In [ ]:
rotation.grad

In [ ]:
loss

In [ ]:
rotation_matrix

In [ ]:
rotation

In [ ]:
theta_hat.shape

In [ ]:
MeanMat = np.load('/srv/data/lily/data/pnl_modular_5/source/meanMat.npy')
VarMat = np.load('/srv/data/lily/data/pnl_modular_5/source/varMat.npy')

In [ ]:
theta_obs = np.concatenate((MeanMat,VarMat), axis=-1)

In [ ]:
theta_hat = model.obs_embed_func(torch.LongTensor([0,1,2,3,4])).detach().numpy()

In [ ]:
plt.figure(figsize=(2,2))
plt.scatter(theta_obs[:,0], theta_hat[:,0])
plt.xlabel('Mean')
plt.ylabel(r'$\theta_{1k}^{obs}$')

In [ ]:
plt.figure(figsize=(2,2))
plt.scatter(theta_obs[:,1], theta_hat[:,1])
plt.xlabel('Var')
plt.ylabel(r'$\theta_{2k}^{obs}$')

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
# ax.set_aspect("equal")
plt.tight_layout()
ax.plot(theta_hat[:,0], theta_hat[:,1], color='red', label='Estimated')
ax.plot(theta_obs[:,0], theta_obs[:,1], color='gray', label='Matrix entries')
arrow(theta_hat[:,0], theta_hat[:,1], ax, 3)
arrow(theta_obs[:,0], theta_obs[:,1], ax, 3)
plt.legend()
plt.grid()
plt.xlabel(r'$\theta_{k1}^{dyn}$')
plt.ylabel(r'$\theta_{k2}^{dyn}$')
plt.title('Trace of change factors')
def arrow(x,y,ax,n):
    d = len(x)//(n+1)    
    ind = np.arange(d,len(x),d)
    for i in ind:
        ar = FancyArrowPatch ((x[i-1],y[i-1]),(x[i],y[i]), 
                              arrowstyle='->', mutation_scale=20)
        ax.add_patch(ar)